# 05 — IBM AMLworld HI-Small EDA (D2, synthetic ground truth)

Week-1 goals (§4.3 D2): schema, laundering prevalence, the eight ground-truth pattern types (these calibrate the §4.4 synthetic motif injector), and the **post-date-range caveat** (Kaggle discussion #427517: transactions after the primary window are all laundering — a leakage trap for naive temporal splits).

Kaggle-metadata expectations for HI-Small: ~5M transactions, ~515K accounts, ~5.1K laundering transactions (rate ≈ 1 per 981), date range Sep 1–10, 2022. License: CDLA-Sharing-1.0.

In [1]:
import polars as pl

RESULTS = []

def check(name, expected, actual):
    """Soft verification: record PASS/FAIL, never crash mid-notebook."""
    ok = expected == actual
    RESULTS.append((name, expected, actual, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}: expected={expected!r} actual={actual!r}")
    return ok

def summary():
    print("\n=== VERIFICATION SUMMARY ===")
    for name, exp, act, ok in RESULTS:
        print(f"{'PASS' if ok else 'FAIL'}  {name}: expected={exp!r} actual={act!r}")
    n_fail = sum(1 for r in RESULTS if not r[3])
    print(f"{len(RESULTS) - n_fail}/{len(RESULTS)} checks passed")


In [2]:
DATA = "../data/raw/amlworld_hi_small"
df = pl.scan_csv(f"{DATA}/HI-Small_Trans.csv").collect()
print(df.shape)
print(df.columns)
df.head(3)

(5078345, 11)
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account_duplicated_0', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


Timestamp,From Bank,Account,To Bank,Account_duplicated_0,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
str,i64,str,i64,str,f64,str,f64,str,str,i64
"""2022/09/01 00:20""",10,"""8000EBD30""",10,"""8000EBD30""",3697.34,"""US Dollar""",3697.34,"""US Dollar""","""Reinvestment""",0
"""2022/09/01 00:20""",3208,"""8000F4580""",1,"""8000F5340""",0.01,"""US Dollar""",0.01,"""US Dollar""","""Cheque""",0
"""2022/09/01 00:00""",3209,"""8000F4670""",3209,"""8000F4670""",14675.57,"""US Dollar""",14675.57,"""US Dollar""","""Reinvestment""",0


In [3]:
n_tx = df.height
lab_col = "Is Laundering"
n_launder = df[lab_col].sum()
print(f"transactions: {n_tx:,}")
print(f"laundering transactions: {n_launder:,} "
      f"(1 per {n_tx // max(n_launder, 1)}, {n_launder / n_tx:.4%})")
accounts = pl.concat([
    df.select(pl.col("Account").alias("a")),
    df.select(pl.col("Account_duplicated_0").alias("a")),
]).unique().height
print(f"distinct account ids: {accounts:,}")
check("order of magnitude: ~5M transactions", True, 4_000_000 < n_tx < 8_000_000)
check("order of magnitude: ~5.1K laundering", True, 3_000 < n_launder < 8_000)

transactions: 5,078,345
laundering transactions: 5,177 (1 per 980, 0.1019%)


distinct account ids: 515,080
PASS  order of magnitude: ~5M transactions: expected=True actual=True
PASS  order of magnitude: ~5.1K laundering: expected=True actual=True


True

In [4]:
# Temporal window + the post-window all-laundering artifact (#427517)
ts = df.select(pl.col("Timestamp").str.to_datetime("%Y/%m/%d %H:%M"))
df2 = df.with_columns(ts)
print("date range:", df2["Timestamp"].min(), "->", df2["Timestamp"].max())
import datetime as dt
late = df2.filter(pl.col("Timestamp") >= dt.datetime(2022, 9, 11))
if late.height:
    print(f"transactions after Sep 10: {late.height:,}; "
          f"laundering share among them: {late[lab_col].mean():.1%}")
else:
    print("no transactions after the primary window in this file")

date range: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00
transactions after Sep 10: 1,108; laundering share among them: 59.1%


In [5]:
# The eight ground-truth laundering pattern types (injector calibration targets)
import re
from collections import Counter
text = open(f"{DATA}/HI-Small_Patterns.txt", encoding="utf-8").read()
types = Counter(re.findall(r"BEGIN LAUNDERING ATTEMPT - (\S+)", text))
for t, c in types.most_common():
    print(f"{t:<20} {c} attempts")
check("distinct pattern types (paper: 8)", 8, len(types))

CYCLE:               54 attempts
GATHER-SCATTER:      51 attempts
BIPARTITE            49 attempts
FAN-OUT:             48 attempts
SCATTER-GATHER       44 attempts
STACK                43 attempts
RANDOM:              41 attempts
FAN-IN:              40 attempts
PASS  distinct pattern types (paper: 8): expected=8 actual=8


True

In [6]:
# Payment formats & currencies (evidence-narrative fields, §4.3 D1 caveat contrast)
print(df["Payment Format"].value_counts().sort("count", descending=True))
print(df["Receiving Currency"].value_counts().sort("count", descending=True).head(8))
summary()

shape: (7, 2)
┌────────────────┬─────────┐
│ Payment Format ┆ count   │
│ ---            ┆ ---     │
│ str            ┆ u32     │
╞════════════════╪═════════╡
│ Cheque         ┆ 1864331 │
│ Credit Card    ┆ 1323324 │
│ ACH            ┆ 600797  │
│ Cash           ┆ 490891  │
│ Reinvestment   ┆ 481056  │
│ Wire           ┆ 171855  │
│ Bitcoin        ┆ 146091  │
└────────────────┴─────────┘


shape: (8, 2)
┌────────────────────┬─────────┐
│ Receiving Currency ┆ count   │
│ ---                ┆ ---     │
│ str                ┆ u32     │
╞════════════════════╪═════════╡
│ US Dollar          ┆ 1879341 │
│ Euro               ┆ 1172017 │
│ Swiss Franc        ┆ 237884  │
│ Yuan               ┆ 206551  │
│ Shekel             ┆ 194988  │
│ Rupee              ┆ 192065  │
│ UK Pound           ┆ 181255  │
│ Ruble              ┆ 157361  │
└────────────────────┴─────────┘

=== VERIFICATION SUMMARY ===
PASS  order of magnitude: ~5M transactions: expected=True actual=True
PASS  order of magnitude: ~5.1K laundering: expected=True actual=True
PASS  distinct pattern types (paper: 8): expected=8 actual=8
3/3 checks passed


**Conclusions.** True amounts/currencies/formats make AMLworld the amount-level-evidence demo dataset (Elliptic's features are anonymized). The eight labeled pattern types calibrate the motif injector (§4.4). The post-window all-laundering artifact must be handled explicitly by the Week-2 temporal splitter (drop or truncate — decision recorded when implemented).